# MatrixEQTL 教学示例：使用自带数据进行 eQTL 分析

本 Notebook 使用 `MatrixEQTL` 包自带的示例数据，演示一个最小但完整的 eQTL 分析流程。

## 本节目标

1. 理解 MatrixEQTL 的输入文件结构；
2. 读取 genotype matrix、expression matrix、covariate matrix；
3. 读取 SNP 位置信息和基因位置信息；
4. 使用 `Matrix_eQTL_main()` 运行 cis/trans eQTL 分析；
5. 查看显著结果、P 值分布、QQ plot 和 top eQTL 结果。

## 适用场景

本 Notebook 用于教学入门。真实项目中通常需要自己准备：

- SNP dosage / genotype matrix；
- gene expression matrix；
- sample covariates；
- SNP genomic position；
- gene genomic position；
- 合理的样本 QC、基因过滤、SNP 过滤、批次校正和群体结构校正。


## 1. 安装和加载 R 包

如果当前环境没有安装 `MatrixEQTL`，先运行安装命令。  
如果已经安装，可以跳过安装代码。

In [1]:
# 如未安装 MatrixEQTL，取消下一行注释后运行
# install.packages("MatrixEQTL")

suppressPackageStartupMessages({
  library(MatrixEQTL)
})

packageVersion("MatrixEQTL")

[1] ‘2.3’

## 2. 查看 MatrixEQTL 自带示例数据

`MatrixEQTL` 包自带了一组小型示例数据。  
这些数据非常适合教学演示，因为不需要额外下载文件。

常见文件包括：

| 文件 | 含义 |
|---|---|
| `SNP.txt` | SNP genotype/dosage 矩阵 |
| `GE.txt` | gene expression 矩阵 |
| `Covariates.txt` | 协变量矩阵 |
| `snpsloc.txt` | SNP 位置信息 |
| `geneloc.txt` | 基因位置信息 |


In [2]:
# 获取示例数据路径
SNP_file_name <- system.file("data/SNP.txt", package = "MatrixEQTL")
expression_file_name <- system.file("data/GE.txt", package = "MatrixEQTL")
covariates_file_name <- system.file("data/Covariates.txt", package = "MatrixEQTL")

snps_location_file_name <- system.file("data/snpsloc.txt", package = "MatrixEQTL")
gene_location_file_name <- system.file("data/geneloc.txt", package = "MatrixEQTL")

example_files <- data.frame(
  file_type = c("SNP matrix", "Expression matrix", "Covariates", "SNP location", "Gene location"),
  path = c(SNP_file_name, expression_file_name, covariates_file_name,
           snps_location_file_name, gene_location_file_name)
)

example_files

file_type,path
<chr>,<chr>
SNP matrix,/home/stereonote/.conda/envs/R443/lib/R/library/MatrixEQTL/data/SNP.txt
Expression matrix,/home/stereonote/.conda/envs/R443/lib/R/library/MatrixEQTL/data/GE.txt
Covariates,/home/stereonote/.conda/envs/R443/lib/R/library/MatrixEQTL/data/Covariates.txt
SNP location,/home/stereonote/.conda/envs/R443/lib/R/library/MatrixEQTL/data/snpsloc.txt
Gene location,/home/stereonote/.conda/envs/R443/lib/R/library/MatrixEQTL/data/geneloc.txt


## 3. 预览输入文件格式

MatrixEQTL 对输入格式有明确要求：

- 第一列通常是 feature ID，例如 SNP ID 或 gene ID；
- 后面的列是样本；
- SNP 矩阵和表达矩阵的样本顺序需要一致；
- 协变量矩阵的列也需要与样本对应。

In [3]:
cat("SNP matrix preview:\n")
print(readLines(SNP_file_name, n = 5))

cat("\nExpression matrix preview:\n")
print(readLines(expression_file_name, n = 5))

cat("\nCovariates matrix preview:\n")
print(readLines(covariates_file_name, n = 5))

cat("\nSNP location preview:\n")
print(readLines(snps_location_file_name, n = 5))

cat("\nGene location preview:\n")
print(readLines(gene_location_file_name, n = 5))

SNP matrix preview:
[1] "snpid\tSam_01\tSam_02\tSam_03\tSam_04\tSam_05\tSam_06\tSam_07\tSam_08\tSam_09\tSam_10\tSam_11\tSam_12\tSam_13\tSam_14\tSam_15\tSam_16"
[2] "Snp_01\t2\t0\t2\t0\t2\t1\t2\t1\t1\t1\t2\t2\t1\t2\t2\t1"                                                                               
[3] "Snp_02\t0\t1\t1\t2\t2\t1\t0\t0\t0\t1\t1\t1\t1\t0\t1\t1"                                                                               
[4] "Snp_03\t1\t0\t1\t0\t1\t1\t1\t1\t0\t1\t1\t0\t1\t1\t1\t2"                                                                               
[5] "Snp_04\t0\t1\t2\t2\t2\t1\t1\t0\t0\t0\t1\t2\t1\t1\t1\t0"                                                                               

Expression matrix preview:
[1] "geneid\tSam_01\tSam_02\tSam_03\tSam_04\tSam_05\tSam_06\tSam_07\tSam_08\tSam_09\tSam_10\tSam_11\tSam_12\tSam_13\tSam_14\tSam_15\tSam_16"
[2] "Gene_01\t4.91\t4.63\t5.18\t5.07\t5.74\t5.09\t5.31\t5.29\t4.73\t5.72\t4.75\t4.54\t5.01\t5.03\t4.84\t4.44"  

## 4. 读取 SNP、表达和协变量矩阵

MatrixEQTL 推荐使用 `SlicedData` 类读取大矩阵。

`SlicedData` 的优点是：

- 适合处理大规模 SNP × sample 或 gene × sample 矩阵；
- 可以分片读取，降低内存压力；
- 是 `Matrix_eQTL_main()` 的标准输入对象。

这里的关键参数：

| 参数 | 含义 |
|---|---|
| `fileDelimiter` | 文件分隔符，通常为 tab |
| `fileSkipRows` | 跳过表头行数 |
| `fileSkipColumns` | 跳过 ID 列数 |
| `fileOmitCharacters` | 缺失值字符 |
| `fileSliceSize` | 每次读取多少行 |


In [5]:
# 读取 SNP genotype/dosage matrix
snps <- SlicedData$new()
snps$fileDelimiter <- "\t"
snps$fileOmitCharacters <- "NA"
snps$fileSkipRows <- 1
snps$fileSkipColumns <- 1
snps$fileSliceSize <- 2000
snps$LoadFile(SNP_file_name)

# 读取 gene expression matrix
gene <- SlicedData$new()
gene$fileDelimiter <- "\t"
gene$fileOmitCharacters <- "NA"
gene$fileSkipRows <- 1
gene$fileSkipColumns <- 1
gene$fileSliceSize <- 2000
gene$LoadFile(expression_file_name)

# 读取 covariate matrix
cvrt <- SlicedData$new()
cvrt$fileDelimiter <- "\t"
cvrt$fileOmitCharacters <- "NA"
cvrt$fileSkipRows <- 1
cvrt$fileSkipColumns <- 1
cvrt$fileSliceSize <- 2000
cvrt$LoadFile(covariates_file_name)

# 为了教学展示维度，直接读取原始小文件查看矩阵大小
snp_preview_mat <- read.table(
  SNP_file_name,
  header = TRUE,
  row.names = 1,
  sep = "\t",
  check.names = FALSE
)

gene_preview_mat <- read.table(
  expression_file_name,
  header = TRUE,
  row.names = 1,
  sep = "\t",
  check.names = FALSE
)

covariate_preview_mat <- read.table(
  covariates_file_name,
  header = TRUE,
  row.names = 1,
  sep = "\t",
  check.names = FALSE
)

cat("SNP matrix dimension:", dim(snp_preview_mat), "\n")
cat("Expression matrix dimension:", dim(gene_preview_mat), "\n")
cat("Covariate matrix dimension:", dim(covariate_preview_mat), "\n")

Rows read: 15 done.

Rows read: 10 done.

Rows read: 2 done.



SNP matrix dimension: 15 16 
Expression matrix dimension: 10 16 
Covariate matrix dimension: 2 16 


## 5. 读取 SNP 和基因位置信息

cis-eQTL 和 trans-eQTL 的划分依赖位置信息。

一般定义：

- **cis-eQTL**：SNP 距离目标基因较近，例如 ±1 Mb；
- **trans-eQTL**：SNP 距离目标基因较远，或者位于不同染色体。

示例数据中已经提供了 SNP 和 gene 的位置文件。

In [6]:
snpspos <- read.table(
  snps_location_file_name,
  header = TRUE,
  stringsAsFactors = FALSE
)

genepos <- read.table(
  gene_location_file_name,
  header = TRUE,
  stringsAsFactors = FALSE
)

cat("SNP position table:\n")
print(head(snpspos))

cat("\nGene position table:\n")
print(head(genepos))

cat("\nSNP position dimension:", dim(snpspos), "\n")
cat("Gene position dimension:", dim(genepos), "\n")

SNP position table:
   snpid  chr    pos
1 Snp_01 chr1 721289
2 Snp_02 chr1 752565
3 Snp_03 chr1 777121
4 Snp_04 chr1 785988
5 Snp_05 chr1 792479
6 Snp_06 chr1 798958

Gene position table:
   geneid  chr   left  right
1 Gene_01 chr1 721289 731289
2 Gene_02 chr1 752565 762565
3 Gene_03 chr1 777121 787121
4 Gene_04 chr1 785988 795988
5 Gene_05 chr1 792479 802479
6 Gene_06 chr1 798958 808958

SNP position dimension: 15 3 
Gene position dimension: 10 4 


## 6. 设置 MatrixEQTL 分析参数

本示例使用线性模型：

\[
\text{Expression} = \beta_0 + \beta_1 \times \text{Genotype} + \beta_2 \times \text{Covariates} + \epsilon
\]

其中：

- Genotype 是 SNP dosage 或 genotype；
- Expression 是基因表达量；
- Covariates 是协变量，例如 batch、sex、PC 等；
- 核心检验是 SNP genotype 是否与 gene expression 显著相关。

常用模型包括：

| 模型 | 含义 |
|---|---|
| `modelLINEAR` | 普通线性模型，最常用 |
| `modelANOVA` | ANOVA 模型 |
| `modelLINEAR_CROSS` | 含交互项的线性模型 |


In [7]:
# 选择模型
useModel <- modelLINEAR

# cis 距离阈值：1 Mb
cisDist <- 1e6

# P 值输出阈值
# 教学示例中阈值可以设得宽一些，便于看到结果
pvOutputThreshold_cis <- 2e-2
pvOutputThreshold_trans <- 1e-2

# 输出文件
outdir <- "/data/work/matrixeqtl_demo"
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

output_file_name_cis <- file.path(outdir, "MatrixEQTL_cis_results.tsv")
output_file_name_trans <- file.path(outdir, "MatrixEQTL_trans_results.tsv")

# errorCovariance 通常用于建模样本间相关性
# 普通独立样本分析中设为 numeric()
errorCovariance <- numeric()

list(
  model = "modelLINEAR",
  cisDist = cisDist,
  pvOutputThreshold_cis = pvOutputThreshold_cis,
  pvOutputThreshold_trans = pvOutputThreshold_trans,
  outdir = outdir
)

$model
[1] "modelLINEAR"

$cisDist
[1] 1e+06

$pvOutputThreshold_cis
[1] 0.02

$pvOutputThreshold_trans
[1] 0.01

$outdir
[1] "/data/work/matrixeqtl_demo"

## 7. 运行 MatrixEQTL

`Matrix_eQTL_main()` 是 MatrixEQTL 的核心函数。

主要输入：

| 参数 | 含义 |
|---|---|
| `snps` | SNP 矩阵 |
| `gene` | 表达矩阵 |
| `cvrt` | 协变量矩阵 |
| `snpspos` | SNP 位置信息 |
| `genepos` | 基因位置信息 |
| `cisDist` | cis 距离阈值 |
| `pvOutputThreshold.cis` | cis 结果输出阈值 |
| `pvOutputThreshold` | trans 结果输出阈值 |


In [8]:
me <- Matrix_eQTL_main(
  snps = snps,
  gene = gene,
  cvrt = cvrt,
  output_file_name = output_file_name_trans,
  pvOutputThreshold = pvOutputThreshold_trans,
  useModel = useModel,
  errorCovariance = errorCovariance,
  verbose = TRUE,
  output_file_name.cis = output_file_name_cis,
  pvOutputThreshold.cis = pvOutputThreshold_cis,
  snpspos = snpspos,
  genepos = genepos,
  cisDist = cisDist,
  pvalue.hist = "qqplot",
  min.pv.by.genesnp = FALSE,
  noFDRsaveMemory = FALSE
)

me

Matching data files and location files

10 of 10 genes matched

15 of 15 SNPs matched


Task finished in 0.008 seconds

Processing covariates

Task finished in 0.006 seconds

Processing gene expression data (imputation, residualization)

Task finished in 0.05 seconds

Creating output file(s)

Task finished in 0.036 seconds

Performing eQTL analysis

100.00% done, 2 cis-eQTLs, 3 trans-eQTLs

Task finished in 0.025 seconds





snps,gene,statistic,pvalue,FDR,beta
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
Snp_13,Gene_09,-3.914403,0.002055817,0.1027908,-0.2978847
Snp_11,Gene_06,-3.221962,0.007327756,0.1619451,-0.2332470
Snp_14,Gene_01,3.070005,0.009716705,0.1619451,0.2147077
snps,gene,statistic,pvalue,FDR,beta
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
Snp_05,Gene_03,38.812160,5.515519e-14,5.515519e-12,0.4101317
Snp_04,Gene_10,3.201666,7.608981e-03,3.804491e-01,0.2321123


## 8. 查看结果概览

MatrixEQTL 结果对象中通常包含：

- `me$cis$eqtls`：cis-eQTL 结果表；
- `me$trans$eqtls`：trans-eQTL 结果表；
- `me$cis$ntests`：cis 检验数量；
- `me$trans$ntests`：trans 检验数量；
- `me$time.in.sec`：运行时间。

In [9]:
cat("Number of cis tests:", me$cis$ntests, "\n")
cat("Number of trans tests:", me$trans$ntests, "\n")
cat("Running time:", me$time.in.sec, "seconds\n")

cat("\nNumber of reported cis eQTLs:", nrow(me$cis$eqtls), "\n")
cat("Number of reported trans eQTLs:", nrow(me$trans$eqtls), "\n")

Number of cis tests: 100 
Number of trans tests: 50 
Running time: 0.112 seconds

Number of reported cis eQTLs: 2 
Number of reported trans eQTLs: 3 


## 9. 查看 cis-eQTL 结果

结果表常见字段：

| 字段 | 含义 |
|---|---|
| `snps` | SNP ID |
| `gene` | gene ID |
| `statistic` | 统计量 |
| `pvalue` | 原始 P 值 |
| `FDR` | 多重检验校正后的 FDR |
| `beta` | 效应值，表示 genotype 对 expression 的影响方向和大小 |


In [10]:
cis_eqtls <- me$cis$eqtls

if (nrow(cis_eqtls) > 0) {
  cis_eqtls <- cis_eqtls[order(cis_eqtls$pvalue), ]
  head(cis_eqtls, 10)
} else {
  cat("No cis-eQTLs reported under current threshold.\n")
}

,snps,gene,statistic,pvalue,FDR,beta
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,Snp_05,Gene_03,38.812160,5.515519e-14,5.515519e-12,0.4101317
2,Snp_04,Gene_10,3.201666,7.608981e-03,3.804491e-01,0.2321123


## 10. 查看 trans-eQTL 结果

trans-eQTL 检验数量通常远大于 cis-eQTL，因此多重检验压力更大。  
真实研究中，trans-eQTL 通常更难检测，需要更大的样本量和更严格的校正。

In [11]:
trans_eqtls <- me$trans$eqtls

if (nrow(trans_eqtls) > 0) {
  trans_eqtls <- trans_eqtls[order(trans_eqtls$pvalue), ]
  head(trans_eqtls, 10)
} else {
  cat("No trans-eQTLs reported under current threshold.\n")
}

,snps,gene,statistic,pvalue,FDR,beta
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,Snp_13,Gene_09,-3.914403,0.002055817,0.1027908,-0.2978847
2,Snp_11,Gene_06,-3.221962,0.007327756,0.1619451,-0.2332470
3,Snp_14,Gene_01,3.070005,0.009716705,0.1619451,0.2147077


## 11. 保存整理后的结果

虽然 `Matrix_eQTL_main()` 已经把结果写入了输出文件，这里再额外保存排序后的结果，便于教学查看。

In [12]:
if (exists("cis_eqtls") && nrow(cis_eqtls) > 0) {
  write.table(
    cis_eqtls,
    file = file.path(outdir, "MatrixEQTL_cis_results_sorted.tsv"),
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
  )
}

if (exists("trans_eqtls") && nrow(trans_eqtls) > 0) {
  write.table(
    trans_eqtls,
    file = file.path(outdir, "MatrixEQTL_trans_results_sorted.tsv"),
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
  )
}

list.files(outdir, full.names = TRUE)

[1] "/data/work/matrixeqtl_demo/01_MatrixEQTL_demo_notebook.ipynb"  
[2] "/data/work/matrixeqtl_demo/MatrixEQTL_cis_results_sorted.tsv"  
[3] "/data/work/matrixeqtl_demo/MatrixEQTL_cis_results.tsv"         
[4] "/data/work/matrixeqtl_demo/MatrixEQTL_trans_results_sorted.tsv"
[5] "/data/work/matrixeqtl_demo/MatrixEQTL_trans_results.tsv"

## 12. 教学总结

通过本 Notebook，我们完成了一个最小 eQTL 分析流程：

1. 读取 genotype、expression 和 covariate 矩阵；
2. 读取 SNP 和 gene 位置信息；
3. 设置 cis 距离阈值；
4. 使用线性模型检测 SNP-expression association；
5. 区分 cis-eQTL 和 trans-eQTL；
6. 查看 top association；
7. 对结果进行基本可视化。

## 真实项目中还需要注意

真实 eQTL 分析远比本示例复杂，通常还需要：

- genotype QC：MAF、missing rate、HWE、imputation INFO；
- expression QC：低表达过滤、标准化、批次校正；
- sample matching：确保 genotype 和 expression 样本完全对应；
- covariates：sex、age、batch、platform、genotype PCs、expression PCs/PEER factors；
- multiple testing correction：cis 和 trans 分开校正；
- ancestry/population structure：控制群体结构；
- tissue/cell-type specificity：不同组织或细胞类型中 eQTL 可能不同。
